# Assam Pipeline 1 - Siamese U-Net to Aligned Tabular Dataset

This notebook starts after preprocessing and patch creation. It uses the model-ready Assam S1+S2+DEM patch chunks:

```text
/content/drive/MyDrive/PRJ3_DATA/Assam_Pipeline1_Model_Ready/<year>/patches/
  X_s1_pre_*.npy
  X_s1_flood_*.npy
  X_s1_post_*.npy
  X_s2_pre_*.npy
  X_s2_flood_*.npy
  X_s2_post_*.npy
  X_dem_*.npy
  y_mask_*.npy
  patch_metadata_*.csv
```

The notebook is restart-friendly for Colab: it validates chunks before U-Net training, saves U-Net checkpoints, saves per-chunk predictions, and skips prediction chunks that already exist. It stops after creating Assam's aligned tabular dataset; RF/XGBoost training happens later after merging all four states.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Core imports
from pathlib import Path
import gc
import math
import os
import random
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import GroupShuffleSplit

print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))


In [ ]:
# Paths and settings
DRIVE_CANDIDATES = [Path('/content/drive/MyDrive/PRJ3_DATA'), Path('/content/drive/MyDrive/Prj_3_Data')]
DRIVE_ROOT = next((p for p in DRIVE_CANDIDATES if p.exists()), DRIVE_CANDIDATES[0])

DATA_ROOT = DRIVE_ROOT / 'Assam_Pipeline1_Model_Ready'
OUTPUT_ROOT = DRIVE_ROOT / 'Assam_Pipeline1_UNet_Tabular_Outputs'
CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
PRED_DIR = OUTPUT_ROOT / 'unet_predictions'
FEATURE_DIR = OUTPUT_ROOT / 'features'
for p in [OUTPUT_ROOT, CHECKPOINT_DIR, PRED_DIR, FEATURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

YEARS = [2019, 2020, 2021, 2022, 2023]
VAL_YEARS = [2023]  # Keep one full flood year out for a more honest validation.
PATCH_SIZE = 128
BATCH_SIZE = 4      # Use 2 if Colab RAM/GPU memory is tight.
EPOCHS = 25
LEARNING_RATE = 1e-4
FLOOD_CLASS_THRESHOLD = 0.02
USE_MIXED_PRECISION = False
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

if USE_MIXED_PRECISION:
    keras.mixed_precision.set_global_policy('mixed_float16')

print('DATA_ROOT:', DATA_ROOT)
print('OUTPUT_ROOT:', OUTPUT_ROOT)


## 1. Validate patch chunks

This cell builds a manifest of complete, readable chunks. If Colab crashed during preprocessing and left an empty or truncated `.npy`, that chunk is skipped here instead of crashing training.


In [ ]:
REQUIRED_PREFIXES = [
    'X_s1_pre', 'X_s1_flood', 'X_s1_post',
    'X_s2_pre', 'X_s2_flood', 'X_s2_post',
    'X_dem', 'y_mask',
]


def npy_ok(path, expected_ndim=None):
    try:
        arr = np.load(path, mmap_mode='r')
        if expected_ndim is not None and arr.ndim != expected_ndim:
            return False, None, f'ndim {arr.ndim} != {expected_ndim}'
        if arr.size == 0:
            return False, None, 'empty array'
        _ = arr.reshape(-1)[0]
        return True, arr.shape, ''
    except Exception as exc:
        return False, None, str(exc)


def chunk_path(patch_dir, prefix, chunk_id):
    return patch_dir / f'{prefix}_{chunk_id}.npy'


def build_manifest(years):
    rows = []
    problems = []
    for year in years:
        patch_dir = DATA_ROOT / str(year) / 'patches'
        if not patch_dir.exists():
            problems.append({'year': year, 'chunk_id': None, 'problem': f'missing {patch_dir}'})
            continue
        y_files = sorted(patch_dir.glob('y_mask_*.npy'), key=lambda p: int(p.stem.split('_')[-1]))
        if not y_files:
            problems.append({'year': year, 'chunk_id': None, 'problem': 'no y_mask chunks'})
            continue
        for y_file in y_files:
            chunk_id = int(y_file.stem.split('_')[-1])
            paths = {prefix: chunk_path(patch_dir, prefix, chunk_id) for prefix in REQUIRED_PREFIXES}
            missing = [prefix for prefix, path in paths.items() if not path.exists()]
            if missing:
                problems.append({'year': year, 'chunk_id': chunk_id, 'problem': 'missing ' + ', '.join(missing)})
                continue
            shapes = {}
            bad = []
            for prefix, path in paths.items():
                ok, shape, err = npy_ok(path, expected_ndim=4)
                if not ok:
                    bad.append(f'{prefix}: {err}')
                else:
                    shapes[prefix] = shape
            if bad:
                problems.append({'year': year, 'chunk_id': chunk_id, 'problem': '; '.join(bad)})
                continue
            counts = {prefix: shape[0] for prefix, shape in shapes.items()}
            if len(set(counts.values())) != 1:
                problems.append({'year': year, 'chunk_id': chunk_id, 'problem': f'count mismatch {counts}'})
                continue
            rows.append({
                'year': year,
                'chunk_id': chunk_id,
                'patches': counts['y_mask'],
                'patch_dir': str(patch_dir),
                **{prefix: str(path) for prefix, path in paths.items()},
            })
    return pd.DataFrame(rows), pd.DataFrame(problems)

manifest, manifest_problems = build_manifest(YEARS)
manifest_path = OUTPUT_ROOT / 'valid_chunk_manifest.csv'
manifest.to_csv(manifest_path, index=False)
manifest_problems.to_csv(OUTPUT_ROOT / 'skipped_chunk_manifest.csv', index=False)

print('Valid chunks:', len(manifest), 'patches:', int(manifest['patches'].sum()) if len(manifest) else 0)
if len(manifest):
    display(manifest.groupby('year')['patches'].sum().reset_index())
if len(manifest_problems):
    print('Skipped chunks/problems:')
    display(manifest_problems)
if manifest.empty:
    raise FileNotFoundError('No complete S1+S2+DEM patch chunks found. Finish preprocessing Cell 9 first.')


## 2. Chunked data loader

The loader opens one patch chunk at a time with memory maps, prepares fixed-size blocks, serves batches from the current block, and asynchronously prepares the next block. The training and validation datasets use `.repeat()` with fixed `steps_per_epoch`/`validation_steps`, so Keras always has data at every epoch boundary without retaining chunks in RAM.

Buffering is intentionally capped for Colab Free stability: no persistent chunk cache, no full-chunk materialization, at most two prepared blocks in RAM, and only one prefetched batch.


In [ ]:
def chw_to_hwc(x):
    return np.moveaxis(x, 1, -1)


# Loader tuning for Colab Free stability: two prepared blocks plus one queued batch only.
INPUT_CHANNELS = 15  # S1 4 + S2 10 + DEM 1 per branch.
CHUNK_BLOCK_SIZE = 384  # Use 512 if Colab RAM is stable; 384 is safer on free runtimes.
CHUNK_CACHE_SIZE = 0      # Keep at 0 on Colab Free; chunk caches can grow RAM across epochs.
CHUNK_PRELOAD = True      # Double-buffer blocks, not full chunks.
PREFETCH_BUFFER = 1       # Avoid AUTOTUNE here; it can increase host RAM buffering.


def load_chunk_arrays(row, mmap_mode='r'):
    s1_pre = np.load(row['X_s1_pre'], mmap_mode=mmap_mode)
    s1_flood = np.load(row['X_s1_flood'], mmap_mode=mmap_mode)
    s2_pre = np.load(row['X_s2_pre'], mmap_mode=mmap_mode)
    s2_flood = np.load(row['X_s2_flood'], mmap_mode=mmap_mode)
    dem = np.load(row['X_dem'], mmap_mode=mmap_mode)
    y = np.load(row['y_mask'], mmap_mode=mmap_mode)
    return s1_pre, s1_flood, s2_pre, s2_flood, dem, y


def chunk_key(row):
    return (int(row['year']), int(row['chunk_id']))


def open_chunk_memmaps(row):
    return {
        's1_pre': np.load(row['X_s1_pre'], mmap_mode='r'),
        's1_flood': np.load(row['X_s1_flood'], mmap_mode='r'),
        's2_pre': np.load(row['X_s2_pre'], mmap_mode='r'),
        's2_flood': np.load(row['X_s2_flood'], mmap_mode='r'),
        'dem': np.load(row['X_dem'], mmap_mode='r'),
        'y': np.load(row['y_mask'], mmap_mode='r'),
    }


def make_block_ranges(n, block_size=CHUNK_BLOCK_SIZE):
    return [(start, min(start + block_size, n)) for start in range(0, n, block_size)]


def load_prepared_block(arrays, start, stop, rng=None):
    """Prepare one contiguous block as NHWC tensors; never materializes a full chunk."""
    sl = slice(start, stop)
    n = stop - start
    pre = np.empty((n, PATCH_SIZE, PATCH_SIZE, INPUT_CHANNELS), dtype=np.float32)
    flood = np.empty_like(pre)

    s1_pre = np.asarray(arrays['s1_pre'][sl], dtype=np.float32)
    pre[..., 0:4] = chw_to_hwc(s1_pre)
    del s1_pre

    s2_pre = np.asarray(arrays['s2_pre'][sl], dtype=np.float32)
    pre[..., 4:14] = chw_to_hwc(s2_pre)
    del s2_pre

    dem_hwc = chw_to_hwc(np.asarray(arrays['dem'][sl], dtype=np.float32))
    pre[..., 14:15] = dem_hwc

    s1_flood = np.asarray(arrays['s1_flood'][sl], dtype=np.float32)
    flood[..., 0:4] = chw_to_hwc(s1_flood)
    del s1_flood

    s2_flood = np.asarray(arrays['s2_flood'][sl], dtype=np.float32)
    flood[..., 4:14] = chw_to_hwc(s2_flood)
    del s2_flood

    flood[..., 14:15] = dem_hwc
    del dem_hwc

    y = np.ascontiguousarray(chw_to_hwc(np.asarray(arrays['y'][sl], dtype=np.float32)))

    order = rng.permutation(n) if rng is not None and n > 1 else np.arange(n)
    return pre, flood, y, order


def count_batches(manifest_df, batch_size):
    return int(sum(math.ceil(int(patches) / batch_size) for patches in manifest_df['patches']))


def chunk_batch_generator(manifest_df, batch_size=4, shuffle=True, seed=42, cache_size=0, preload=True, block_size=CHUNK_BLOCK_SIZE):
    manifest_records = manifest_df.reset_index(drop=True).to_dict('records')
    if cache_size:
        print('Ignoring CHUNK_CACHE_SIZE for stable Colab training; blocks are released after use.')
    epoch = 0

    def ordered_positions(rng):
        row_order = np.arange(len(manifest_records))
        if shuffle:
            rng.shuffle(row_order)
        return row_order

    def block_ranges_for_chunk(n, rng):
        ranges = make_block_ranges(n, block_size=block_size)
        if shuffle and len(ranges) > 1:
            order = rng.permutation(len(ranges))
            ranges = [ranges[int(i)] for i in order]
        return ranges

    def iter_prepared_blocks(arrays, ranges, rng):
        if not preload or len(ranges) <= 1:
            for start, stop in ranges:
                yield load_prepared_block(arrays, start, stop, rng if shuffle else None)
            return

        from concurrent.futures import ThreadPoolExecutor
        with ThreadPoolExecutor(max_workers=1) as executor:
            next_i = 0
            start, stop = ranges[next_i]
            future = executor.submit(load_prepared_block, arrays, start, stop, rng if shuffle else None)
            next_i += 1
            while future is not None:
                block = future.result()
                if next_i < len(ranges):
                    start, stop = ranges[next_i]
                    future = executor.submit(load_prepared_block, arrays, start, stop, rng if shuffle else None)
                    next_i += 1
                else:
                    future = None
                yield block

    def generator():
        nonlocal epoch
        rng = np.random.default_rng(seed + epoch)
        epoch += 1
        for row_pos in ordered_positions(rng):
            arrays = None
            try:
                row = manifest_records[int(row_pos)]
                arrays = open_chunk_memmaps(row)
                ranges = block_ranges_for_chunk(int(row['patches']), rng)
                for pre, flood, y, order in iter_prepared_blocks(arrays, ranges, rng):
                    try:
                        for start in range(0, len(order), batch_size):
                            batch_idx = order[start:start + batch_size]
                            batch_pre = np.ascontiguousarray(pre[batch_idx], dtype=np.float32)
                            batch_flood = np.ascontiguousarray(flood[batch_idx], dtype=np.float32)
                            batch_y = np.ascontiguousarray(y[batch_idx], dtype=np.float32)
                            yield {'pre': batch_pre, 'flood': batch_flood}, batch_y
                            del batch_idx, batch_pre, batch_flood, batch_y
                    finally:
                        del pre, flood, y, order
                        gc.collect()
            finally:
                if arrays is not None:
                    arrays.clear()
                del arrays
                gc.collect()

    return generator


def make_chunk_dataset(manifest_df, batch_size=4, shuffle=True, seed=42, cache_size=0, preload=True, repeat=True):
    signature = (
        {
            'pre': tf.TensorSpec(shape=(None, PATCH_SIZE, PATCH_SIZE, INPUT_CHANNELS), dtype=tf.float32),
            'flood': tf.TensorSpec(shape=(None, PATCH_SIZE, PATCH_SIZE, INPUT_CHANNELS), dtype=tf.float32),
        },
        tf.TensorSpec(shape=(None, PATCH_SIZE, PATCH_SIZE, 1), dtype=tf.float32),
    )
    ds = tf.data.Dataset.from_generator(
        chunk_batch_generator(manifest_df, batch_size=batch_size, shuffle=shuffle, seed=seed, cache_size=cache_size, preload=preload),
        output_signature=signature,
    )
    options = tf.data.Options()
    options.experimental_deterministic = not shuffle
    ds = ds.with_options(options)
    if repeat:
        ds = ds.repeat()
    ds = ds.prefetch(PREFETCH_BUFFER)
    return ds


train_manifest = manifest[~manifest['year'].isin(VAL_YEARS)].reset_index(drop=True)
val_manifest = manifest[manifest['year'].isin(VAL_YEARS)].reset_index(drop=True)
if val_manifest.empty:
    print('VAL_YEARS not available; using a grouped random split by year/chunk.')
    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
    train_idx, val_idx = next(splitter.split(manifest, groups=manifest['year'].astype(str) + '_' + manifest['chunk_id'].astype(str)))
    train_manifest = manifest.iloc[train_idx].reset_index(drop=True)
    val_manifest = manifest.iloc[val_idx].reset_index(drop=True)

TRAIN_STEPS = count_batches(train_manifest, BATCH_SIZE)
VAL_STEPS = count_batches(val_manifest, BATCH_SIZE)
train_seq = make_chunk_dataset(train_manifest, batch_size=BATCH_SIZE, shuffle=True, seed=SEED, cache_size=CHUNK_CACHE_SIZE, preload=CHUNK_PRELOAD)
val_seq = make_chunk_dataset(val_manifest, batch_size=BATCH_SIZE, shuffle=False, seed=SEED, cache_size=CHUNK_CACHE_SIZE, preload=CHUNK_PRELOAD)

sample_preview = make_chunk_dataset(train_manifest.head(1), batch_size=BATCH_SIZE, shuffle=False, seed=SEED, cache_size=0, preload=False, repeat=False)
sample_x, sample_y = next(iter(sample_preview.take(1)))
print('Input channels per branch:', INPUT_CHANNELS)
print('Train patches:', int(train_manifest['patches'].sum()), 'Val patches:', int(val_manifest['patches'].sum()))
print('Train steps:', TRAIN_STEPS, 'Val steps:', VAL_STEPS)
print('Batch:', sample_x['pre'].shape, sample_x['flood'].shape, sample_y.shape)
del sample_preview, sample_x, sample_y
gc.collect()


## 3. Siamese U-Net

The same encoder processes the pre-flood and flood-date inputs. The decoder receives the flood features plus absolute pre/flood feature differences, which makes the model focus on flood-related change while still using S2 and DEM context.


In [ ]:
def conv_block(x, filters, name):
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f'{name}_conv1')(x)
    x = layers.BatchNormalization(name=f'{name}_bn1')(x)
    x = layers.Activation('relu', name=f'{name}_relu1')(x)
    x = layers.Conv2D(filters, 3, padding='same', use_bias=False, name=f'{name}_conv2')(x)
    x = layers.BatchNormalization(name=f'{name}_bn2')(x)
    x = layers.Activation('relu', name=f'{name}_relu2')(x)
    return x


def build_shared_encoder(input_shape=(128, 128, 15), base=24):
    inp = keras.Input(shape=input_shape)
    c1 = conv_block(inp, base, 'enc1')
    p1 = layers.MaxPool2D()(c1)
    c2 = conv_block(p1, base * 2, 'enc2')
    p2 = layers.MaxPool2D()(c2)
    c3 = conv_block(p2, base * 4, 'enc3')
    p3 = layers.MaxPool2D()(c3)
    c4 = conv_block(p3, base * 8, 'enc4')
    p4 = layers.MaxPool2D()(c4)
    b = conv_block(p4, base * 12, 'bottleneck')
    return keras.Model(inp, [c1, c2, c3, c4, b], name='shared_encoder')


def up_block(x, skip, filters, name):
    x = layers.UpSampling2D(interpolation='bilinear', name=f'{name}_up')(x)
    x = layers.Concatenate(name=f'{name}_concat')([x, skip])
    x = conv_block(x, filters, name)
    return x


def dice_coef(y_true, y_pred, smooth=1.0):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=[1, 2, 3])
    denom = tf.reduce_sum(y_true + y_pred, axis=[1, 2, 3])
    return tf.reduce_mean((2.0 * intersection + smooth) / (denom + smooth))


def bce_dice_loss(y_true, y_pred):
    bce = keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + (1.0 - dice_coef(y_true, y_pred))


def abs_diff(xs):
    return tf.abs(xs[0] - xs[1])


def first_input_shape(shapes):
    return shapes[0]


def build_siamese_unet(input_channels, base=24):
    pre_in = keras.Input((PATCH_SIZE, PATCH_SIZE, input_channels), name='pre')
    flood_in = keras.Input((PATCH_SIZE, PATCH_SIZE, input_channels), name='flood')
    encoder = build_shared_encoder((PATCH_SIZE, PATCH_SIZE, input_channels), base=base)
    pre_skips = encoder(pre_in)
    flood_skips = encoder(flood_in)

    diff_b = layers.Lambda(abs_diff, output_shape=first_input_shape, name='bottleneck_absdiff')([flood_skips[-1], pre_skips[-1]])
    x = layers.Concatenate(name='bottleneck_merge')([flood_skips[-1], diff_b])

    for level, filters in [(3, base * 8), (2, base * 4), (1, base * 2), (0, base)]:
        diff_skip = layers.Lambda(abs_diff, output_shape=first_input_shape, name=f'skip{level + 1}_absdiff')([flood_skips[level], pre_skips[level]])
        skip = layers.Concatenate(name=f'skip{level + 1}_merge')([flood_skips[level], diff_skip])
        x = up_block(x, skip, filters, f'dec{level + 1}')

    out = layers.Conv2D(1, 1, activation='sigmoid', dtype='float32', name='flood_probability')(x)
    model = keras.Model({'pre': pre_in, 'flood': flood_in}, out, name='assam_siamese_unet')
    model.compile(
        optimizer=keras.optimizers.Adam(LEARNING_RATE),
        loss=bce_dice_loss,
        metrics=[dice_coef, keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name='binary_iou')],
    )
    return model


UNET_CUSTOM_OBJECTS = {
    'bce_dice_loss': bce_dice_loss,
    'dice_coef': dice_coef,
    'abs_diff': abs_diff,
    'first_input_shape': first_input_shape,
}


def load_unet_checkpoint(model_path):
    try:
        return keras.models.load_model(str(model_path), custom_objects=UNET_CUSTOM_OBJECTS, safe_mode=False)
    except NotImplementedError as exc:
        if 'Lambda' not in str(exc) and 'output_shape' not in str(exc):
            raise
        print('Checkpoint has legacy Lambda layers without output_shape; rebuilding model and loading weights.')
        restored = build_siamese_unet(INPUT_CHANNELS, base=24)
        try:
            restored.load_weights(str(model_path))
        except Exception:
            import tempfile
            import zipfile
            with tempfile.TemporaryDirectory() as tmp_dir:
                with zipfile.ZipFile(model_path, 'r') as zf:
                    zf.extract('model.weights.h5', path=tmp_dir)
                restored.load_weights(str(Path(tmp_dir) / 'model.weights.h5'))
        return restored

model = build_siamese_unet(INPUT_CHANNELS, base=24)
model.summary()


## 4. Train with resume-safe checkpoints

Run this cell again after a Colab crash. `BackupAndRestore` and the checkpoint file will pick up from the last completed epoch when possible.


In [ ]:
checkpoint_path = CHECKPOINT_DIR / 'assam_siamese_unet_best.keras'
backup_dir = CHECKPOINT_DIR / 'backup_and_restore'

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(checkpoint_path),
        monitor='val_dice_coef',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.CSVLogger(str(OUTPUT_ROOT / 'unet_training_log.csv'), append=True),
    keras.callbacks.BackupAndRestore(str(backup_dir)),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-6, verbose=1),
    keras.callbacks.EarlyStopping(monitor='val_dice_coef', mode='max', patience=7, restore_best_weights=True, verbose=1),
]

history = model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=EPOCHS,
    steps_per_epoch=TRAIN_STEPS,
    validation_steps=VAL_STEPS,
    callbacks=callbacks,
)
model.save(str(OUTPUT_ROOT / 'assam_siamese_unet_last.keras'))
print('Saved last model to:', OUTPUT_ROOT / 'assam_siamese_unet_last.keras')
print('Best checkpoint:', checkpoint_path)


## 5. Evaluate and visualize validation patches

In [ ]:
best_model_path = CHECKPOINT_DIR / 'assam_siamese_unet_best.keras'
if best_model_path.exists():
    model = load_unet_checkpoint(best_model_path)
    print('Loaded best model:', best_model_path)

metrics = model.evaluate(val_seq, steps=VAL_STEPS, return_dict=True)
pd.DataFrame([metrics]).to_csv(OUTPUT_ROOT / 'unet_validation_metrics.csv', index=False)
print(metrics)

batch_x, batch_y = next(iter(val_seq.take(1)))
pred = model.predict(batch_x, verbose=0)
flood_preview = batch_x['flood'].numpy()
label_preview = batch_y.numpy()
plt.figure(figsize=(12, 8))
for i in range(min(4, pred.shape[0])):
    plt.subplot(4, 3, i * 3 + 1)
    plt.imshow(flood_preview[i, :, :, 1], cmap='gray')
    plt.title('Flood input VH')
    plt.axis('off')
    plt.subplot(4, 3, i * 3 + 2)
    plt.imshow(label_preview[i, :, :, 0], cmap='Blues', vmin=0, vmax=1)
    plt.title('Pseudo label')
    plt.axis('off')
    plt.subplot(4, 3, i * 3 + 3)
    plt.imshow(pred[i, :, :, 0], cmap='magma', vmin=0, vmax=1)
    plt.title('U-Net probability')
    plt.axis('off')
plt.tight_layout()
plt.show()


## 6. Save U-Net flood masks and patch-level labels

This produces per-chunk probability arrays and a patch table with `pred_flood_fraction`, `label_flood_fraction`, and `is_flood`. Already-created prediction chunks are skipped, so this is safe to resume.


In [ ]:
def predict_chunk(row, model, batch_size=4):
    s1_pre, s1_flood, s2_pre, s2_flood, dem, y = load_chunk_arrays(row)
    n = int(row['patches'])
    probs = []
    for start in range(0, n, batch_size):
        sl = slice(start, min(start + batch_size, n))
        pre = np.concatenate([s1_pre[sl], s2_pre[sl], dem[sl]], axis=1).astype(np.float32)
        flood = np.concatenate([s1_flood[sl], s2_flood[sl], dem[sl]], axis=1).astype(np.float32)
        pred = model.predict({'pre': chw_to_hwc(pre), 'flood': chw_to_hwc(flood)}, verbose=0)
        probs.append(pred.astype(np.float32))
    return np.concatenate(probs, axis=0), np.asarray(y, dtype=np.uint8)

all_prediction_rows = []
for _, row in manifest.iterrows():
    year = int(row['year'])
    chunk_id = int(row['chunk_id'])
    year_pred_dir = PRED_DIR / str(year)
    year_pred_dir.mkdir(parents=True, exist_ok=True)
    prob_path = year_pred_dir / f'unet_prob_{chunk_id}.npy'
    mask_path = year_pred_dir / f'unet_mask_{chunk_id}.npy'
    csv_path = year_pred_dir / f'unet_patch_predictions_{chunk_id}.csv'

    if prob_path.exists() and mask_path.exists() and csv_path.exists():
        print(f'{year} chunk {chunk_id}: prediction exists, skipping')
        chunk_df = pd.read_csv(csv_path)
        all_prediction_rows.append(chunk_df)
        continue

    print(f'Predicting {year} chunk {chunk_id}')
    prob, y_true = predict_chunk(row, model, batch_size=BATCH_SIZE)
    pred_mask = (prob >= 0.5).astype(np.uint8)
    np.save(prob_path, prob.astype(np.float32))
    np.save(mask_path, pred_mask)

    meta_path = Path(row['patch_dir']) / f'patch_metadata_{chunk_id}.csv'
    if meta_path.exists():
        meta = pd.read_csv(meta_path)
    else:
        meta = pd.DataFrame({'local_patch': np.arange(prob.shape[0])})
    meta['year'] = year
    meta['chunk_id'] = chunk_id
    meta['local_patch'] = np.arange(prob.shape[0])
    meta['label_flood_fraction'] = y_true.mean(axis=(1, 2, 3))
    meta['pred_flood_fraction'] = prob.mean(axis=(1, 2, 3))
    meta['is_flood'] = (meta['pred_flood_fraction'] >= FLOOD_CLASS_THRESHOLD).astype(int)
    meta.to_csv(csv_path, index=False)
    all_prediction_rows.append(meta)
    del prob, pred_mask, y_true
    gc.collect()

patch_predictions = pd.concat(all_prediction_rows, ignore_index=True)
patch_predictions.to_csv(OUTPUT_ROOT / 'assam_unet_patch_predictions.csv', index=False)
display(patch_predictions.groupby('year')[['label_flood_fraction', 'pred_flood_fraction', 'is_flood']].mean().reset_index())
print('Saved patch predictions:', OUTPUT_ROOT / 'assam_unet_patch_predictions.csv')


## 7. Build aligned tabular dataset

Features come from S1/S2/DEM patch summaries, U-Net predicted flood fraction, patch row/column, and optional rainfall/river-level CSVs. This is the final Assam output for the later four-state RF/XGBoost notebook.


In [ ]:
# Memory-safe feature extraction from patch chunks.
FEATURE_BATCH_SIZE = 128


def add_channel_stats(df, prefix, arr_path, batch_size=FEATURE_BATCH_SIZE):
    arr = np.load(arr_path, mmap_mode='r')
    n, channels = arr.shape[0], arr.shape[1]
    means = np.empty((n, channels), dtype=np.float32)
    stds = np.empty((n, channels), dtype=np.float32)
    mins = np.empty((n, channels), dtype=np.float32)
    maxs = np.empty((n, channels), dtype=np.float32)
    for start in range(0, n, batch_size):
        sl = slice(start, min(start + batch_size, n))
        block = np.asarray(arr[sl], dtype=np.float32)
        means[sl] = block.mean(axis=(2, 3))
        stds[sl] = block.std(axis=(2, 3))
        mins[sl] = block.min(axis=(2, 3))
        maxs[sl] = block.max(axis=(2, 3))
    for c in range(channels):
        base = f'{prefix}_c{c}'
        df[f'{base}_mean'] = means[:, c]
        df[f'{base}_std'] = stds[:, c]
        df[f'{base}_min'] = mins[:, c]
        df[f'{base}_max'] = maxs[:, c]


def add_diff_stats(df, out_prefix, path_a, path_b, batch_size=FEATURE_BATCH_SIZE):
    arr_a = np.load(path_a, mmap_mode='r')
    arr_b = np.load(path_b, mmap_mode='r')
    n, channels = arr_a.shape[0], arr_a.shape[1]
    means = np.empty((n, channels), dtype=np.float32)
    stds = np.empty((n, channels), dtype=np.float32)
    mins = np.empty((n, channels), dtype=np.float32)
    maxs = np.empty((n, channels), dtype=np.float32)
    for start in range(0, n, batch_size):
        sl = slice(start, min(start + batch_size, n))
        diff = np.asarray(arr_a[sl], dtype=np.float32) - np.asarray(arr_b[sl], dtype=np.float32)
        means[sl] = diff.mean(axis=(2, 3))
        stds[sl] = diff.std(axis=(2, 3))
        mins[sl] = diff.min(axis=(2, 3))
        maxs[sl] = diff.max(axis=(2, 3))
    for c in range(channels):
        base = f'{out_prefix}_c{c}'
        df[f'{base}_mean'] = means[:, c]
        df[f'{base}_std'] = stds[:, c]
        df[f'{base}_min'] = mins[:, c]
        df[f'{base}_max'] = maxs[:, c]


def chunk_feature_frame(row):
    year = int(row['year'])
    chunk_id = int(row['chunk_id'])
    n = int(row['patches'])
    df = pd.DataFrame({'year': year, 'chunk_id': chunk_id, 'local_patch': np.arange(n)})

    for prefix in ['X_s1_pre', 'X_s1_flood', 'X_s1_post', 'X_s2_pre', 'X_s2_flood', 'X_s2_post', 'X_dem']:
        add_channel_stats(df, prefix, row[prefix])

    add_diff_stats(df, 's1_change', row['X_s1_flood'], row['X_s1_pre'])
    add_diff_stats(df, 's2_change', row['X_s2_flood'], row['X_s2_pre'])

    meta_path = Path(row['patch_dir']) / f'patch_metadata_{chunk_id}.csv'
    if meta_path.exists():
        meta = pd.read_csv(meta_path)
        keep_cols = [c for c in ['row', 'col', 'flood_fraction', 'is_empty_kept'] if c in meta.columns]
        if keep_cols:
            df = pd.concat([df, meta[keep_cols].reset_index(drop=True)], axis=1)
    pred_path = PRED_DIR / str(year) / f'unet_patch_predictions_{chunk_id}.csv'
    if pred_path.exists():
        pred_df = pd.read_csv(pred_path)
        for col in ['pred_flood_fraction', 'label_flood_fraction', 'is_flood']:
            if col in pred_df.columns:
                df[col] = pred_df[col].values
    y = np.load(row['y_mask'], mmap_mode='r')
    if 'label_flood_fraction' not in df:
        label_frac = np.empty(n, dtype=np.float32)
        for start in range(0, n, FEATURE_BATCH_SIZE):
            sl = slice(start, min(start + FEATURE_BATCH_SIZE, n))
            label_frac[sl] = np.asarray(y[sl]).mean(axis=(1, 2, 3))
        df['label_flood_fraction'] = label_frac
    if 'is_flood' not in df:
        df['is_flood'] = (df['label_flood_fraction'] >= FLOOD_CLASS_THRESHOLD).astype(int)
    return df

feature_parts = []
for _, row in manifest.iterrows():
    out_path = FEATURE_DIR / f"features_{int(row['year'])}_{int(row['chunk_id'])}.pkl"
    if out_path.exists():
        feature_parts.append(pd.read_pickle(out_path))
        continue
    print('Building features:', row['year'], row['chunk_id'])
    df = chunk_feature_frame(row)
    df.to_pickle(out_path)
    feature_parts.append(df)
    gc.collect()

features = pd.concat(feature_parts, ignore_index=True)
features.to_csv(OUTPUT_ROOT / 'assam_patch_features.csv', index=False)
print('Feature table:', features.shape)
display(features.head())


In [ ]:
def load_hydro_year_features(paths, prefix):
    rows = []
    for path in paths:
        path = Path(path)
        if not path.exists():
            print('Missing hydro file:', path)
            continue
        df = pd.read_csv(path)
        date_cols = [c for c in df.columns if c.lower() in ['date', 'datetime', 'time']]
        if date_cols:
            df[date_cols[0]] = pd.to_datetime(df[date_cols[0]], errors='coerce')
            df['year'] = df[date_cols[0]].dt.year
        elif 'year' not in df.columns:
            print('Hydro file has no date/year column, skipping:', path)
            continue
        numeric = df.select_dtypes(include=[np.number]).copy()
        numeric['year'] = df['year']
        agg = numeric.groupby('year').agg(['mean', 'max', 'min', 'std']).reset_index()
        agg.columns = ['year'] + [f'{prefix}_{a}_{b}' for a, b in agg.columns[1:]]
        rows.append(agg)
    if not rows:
        return pd.DataFrame({'year': YEARS})
    out = rows[0]
    for extra in rows[1:]:
        out = out.merge(extra, on='year', how='outer')
    return out

rain_features = load_hydro_year_features(RAINFALL_CSVS, 'rain')
river_features = load_hydro_year_features(RIVER_LEVEL_CSVS, 'river')
hydro_features = rain_features.merge(river_features, on='year', how='outer')
model_table = features.merge(hydro_features, on='year', how='left')
model_table = model_table.replace([np.inf, -np.inf], np.nan).fillna(0)
model_table['state'] = 'Assam'
front_cols = ['state', 'year', 'chunk_id', 'local_patch']
other_cols = [c for c in model_table.columns if c not in front_cols]
model_table = model_table[front_cols + other_cols]
aligned_path = OUTPUT_ROOT / 'assam_aligned_tabular_dataset.csv'
model_table.to_csv(aligned_path, index=False)
print('Aligned Assam tabular dataset:', model_table.shape)
print('Saved:', aligned_path)
display(model_table.head())


## Notes

- Use this notebook after the preprocessing notebook has completed Cell 9 patch extraction.
- If Colab crashes during U-Net training, rerun from the top and then the training cell. The checkpoint/backup folder should resume from the last completed epoch.
- If Colab crashes during prediction or feature creation, rerun those cells. Existing prediction and feature chunks are reused.
- Add rainfall and river-level CSV paths in Section 7 when you are ready to include hydrological features.
- Stop here for Assam. Do not train RF/XGBoost until Assam, Kerala, and the other states are merged into one tabular dataset.
